# Experimentos

In [1]:
import sys, os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import pandas as pd
import mlflow
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

from data import get_training_data

mlflow.set_tracking_uri("file:" + os.path.join(os.getcwd(), "mlruns"))
mlflow.set_experiment("stock-direction")

2026/06/17 21:41:43 INFO mlflow.tracking.fluent: Experiment with name 'stock-direction' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///Users/alejandroospinasantos/mlops-proyecto-acciones/mlruns/132333012111677087', creation_time=1781725303421, experiment_id='132333012111677087', last_update_time=1781725303421, lifecycle_stage='active', name='stock-direction', tags={}>

## Datos y partición

In [2]:
X, y, feature_names = get_training_data()

# split temporal: 80% entrena, 20% prueba, sin shuffle
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

majority = y_train.mode()[0]
baseline_acc = (y_test == majority).mean()
print("Entrenamiento:", len(X_train), "| Prueba:", len(X_test))
print("Baseline:", round(baseline_acc, 4))

Entrenamiento: 1996 | Prueba: 500
Baseline: 0.562


## Modelos

In [3]:
def entrenar_y_registrar(modelo, nombre, params=None):
    with mlflow.start_run(run_name=nombre):
        pipe = Pipeline([("scaler", StandardScaler()), ("clf", modelo)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        proba = pipe.predict_proba(X_test)[:, 1]
        metrics = {
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, proba),
        }
        mlflow.log_param("modelo", nombre)
        if params:
            mlflow.log_params(params)
        mlflow.log_metric("baseline_accuracy", baseline_acc)
        for k, v in metrics.items():
            mlflow.log_metric(k, v)
        return metrics

In [4]:
m1 = entrenar_y_registrar(
    LogisticRegression(max_iter=1000, C=1.0),
    "logistic_regression",
    {"C": 1.0},
)
m1

{'accuracy': 0.544,
 'precision': 0.563855421686747,
 'recall': 0.8327402135231317,
 'f1': 0.6724137931034483,
 'roc_auc': 0.47176587204861964}

In [5]:
m2 = entrenar_y_registrar(
    RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
    "random_forest",
    {"n_estimators": 200, "max_depth": 5},
)
m2

{'accuracy': 0.552,
 'precision': 0.5728900255754475,
 'recall': 0.797153024911032,
 'f1': 0.6666666666666666,
 'roc_auc': 0.49724564910057034}

In [6]:
m3 = entrenar_y_registrar(
    GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42),
    "gradient_boosting",
    {"n_estimators": 200, "max_depth": 3},
)
m3

{'accuracy': 0.486,
 'precision': 0.5413793103448276,
 'recall': 0.5587188612099644,
 'f1': 0.5499124343257443,
 'roc_auc': 0.46619217081850545}

In [8]:
m4 = entrenar_y_registrar(
    LogisticRegression(max_iter=1000, C=0.1),
    "logistic_regression_c01",
    {"C": 0.1},
)
m4

{'accuracy': 0.54,
 'precision': 0.5611510791366906,
 'recall': 0.8327402135231317,
 'f1': 0.670487106017192,
 'roc_auc': 0.47215586863614945}

In [9]:
m5 = entrenar_y_registrar(
    RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42),
    "random_forest_simple",
    {"n_estimators": 100, "max_depth": 3},
)
m5

{'accuracy': 0.562,
 'precision': 0.5711009174311926,
 'recall': 0.8861209964412812,
 'f1': 0.694560669456067,
 'roc_auc': 0.5050293309933538}

## Comparación

In [10]:
pd.DataFrame(
    [m1, m2, m3, m4, m5],
    index=[
        "logistic_regression",
        "random_forest",
        "gradient_boosting",
        "logistic_regression_c01",
        "random_forest_simple",
    ],
)

,accuracy,precision,recall,f1,roc_auc
logistic_regression,0.544,0.563855,0.832740,0.672414,0.471766
random_forest,0.552,0.572890,0.797153,0.666667,0.497246
gradient_boosting,0.486,0.541379,0.558719,0.549912,0.466192
logistic_regression_c01,0.540,0.561151,0.832740,0.670487,0.472156
random_forest_simple,0.562,0.571101,0.886121,0.694561,0.505029


## Modelo elegido
